In [3]:
import typesense


In [6]:
client = typesense.Client({
    'nodes': [{
        'host': 'l8a5wnxyt2oh9p-1.a1.typesense.net',
        'port': '443',
        'protocol': 'https'
    }],
    'api_key':'kfl7rI0OUu4LrE3ZiiZfhvS1elvM',
  'connection_timeout_seconds': 2
})

books_schema = {
    'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}
print(client.collections.create(books_schema))

ConnectError: [Errno 11001] getaddrinfo failed

In [ ]:
client

NameError: name 'client' is not defined

In [ ]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [ ]:
search_parameters={
    'q':"harry potter",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)


In [ ]:
search_parameters = {
  'q'         : 'harry potter',
  'query_by'  : 'title',
  'filter_by' : 'publication_year:<1998',
  'sort_by'   : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
  'q'         : 'experyment',
  'query_by'  : 'title',
  'facet_by'  : 'authors',
  'sort_by'   : 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [ ]:

import os
os.environ["GROQ_API_KEY"] = ""

In [ ]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

In [ ]:
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "8a5wnxyt2pmjdoh9p-1.a1.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key":"kfl7rI0OUu4LrE3ZnQrZiiZfhvS1elvM",
        "typesense_collection_name": "lang-chain"
    },
    
)

In [ ]:
query = "What is artificial intelligence"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

In [ ]:
### Retriever
retriever = docsearch.as_retriever()
retriever


In [ ]:
query = "Artificial intelligence indepth explanation"
retriever.invoke(query)[0]